# Superstore - Retail

In [1]:
import pandas as pd 
import numpy as np 

In [2]:
data = pd.read_csv("../data/raw/superstore-tableau.csv")

In [ ]:
data.head()

In [ ]:
data.columns

In [3]:
# Creating unique data tables

customer = data[["Customer ID", "Customer Name", "Segment", "Country", "State", "City", "Postal Code", "Region"]].drop_duplicates()
product = data[["Product ID", "Category", "Sub-Category", "Product Name"]].drop_duplicates()
orders = data[["Customer ID", "Order ID", "Order Date", "Ship Date", "Ship Mode", "Product ID", "Sales", "Quantity", "Discount", "Profit"]].drop_duplicates()
orders['Order Date'] = pd.to_datetime(orders['Order Date'], format = "mixed")
orders['Ship Date'] = pd.to_datetime(orders['Ship Date'], format = "mixed")
orders["days_till_shipped"] = (orders["Ship Date"] - orders["Order Date"]).dt.days

dates = pd.DataFrame(orders['Order Date'].unique()).sort_values(by=0).reset_index(drop=True)
dates.columns = ['date']
dates['month'] = dates['date'].dt.month_name()
dates['month_num'] = dates['date'].dt.month
dates['year'] = dates['date'].dt.year
dates['quarter'] = dates['date'].dt.quarter



In [4]:
# creating aggregated view of orders
columns = ['Order ID', 'Order Date', 'year',"quarter",  'month', "month_num",'Ship Mode', 'days_till_shipped', 'Sales',
       'Quantity', 'Product ID' ]
orders_agg = orders.groupby(["Order ID", 
                             "Order Date", 
                             "Ship Mode",
                             "days_till_shipped"])[
                                 ["Sales",
                                  "Quantity"]
                                 ].sum().reset_index().merge(
                                     orders.groupby("Order ID")["Product ID"].count(),
                                      on="Order ID").merge(dates[["date",
                                                        "year","quarter", "month","month_num"]], 
                                                        left_on="Order Date", 
                                                        right_on = "date")[columns]
orders_agg = orders_agg.sort_values('Order Date').reset_index(drop=True)
orders_agg.columns = ['order_id', 'order_date', 'year',"quarter",'month', "month_num",'ship_mode', 'days_till_shipped', 'revenue',
       'quantity', 'products' ]
orders_agg

,order_id,order_date,year,quarter,month,month_num,ship_mode,days_till_shipped,revenue,quantity,products
0,CA-2014-140795,2014-01-02,2014,1,January,1,First Class,59,468.900,6,1
1,CA-2014-104269,2014-01-03,2014,1,January,1,Second Class,151,457.568,2,1
2,CA-2014-168312,2014-01-03,2014,1,January,1,Standard Class,181,513.861,6,2
3,CA-2014-113880,2014-01-03,2014,1,January,1,Standard Class,120,651.588,9,2
4,US-2014-143707,2014-01-03,2014,1,January,1,Standard Class,120,5.940,3,1
...,...,...,...,...,...,...,...,...,...,...,...
5004,US-2017-158526,2017-12-29,2017,4,December,12,Second Class,3,1814.680,14,5
5005,CA-2017-156720,2017-12-30,2017,4,December,12,Standard Class,61,3.024,3,1
5006,CA-2017-143259,2017-12-30,2017,4,December,12,Standard Class,61,466.842,14,3
5007,CA-2017-115427,2017-12-30,2017,4,December,12,Standard Class,61,34.624,4,2


In [5]:
# Let's check total  orders
print("Total orders:", len(orders_agg['order_id']))

# Let's check total  sales
print("Total sales:", round(orders_agg['revenue'].sum(), 2))

# Let's check total  customers
print("Total customers:", len(orders['Customer ID'].unique()))


Total orders: 5009
Total sales: 2296919.49
Total customers: 793


In [6]:
# Total orders, sales, products, quantities through 2014-2017
(orders_agg.groupby(["year"])["order_id"].count().reset_index()).join(
    orders_agg.groupby(["year"])["revenue"].sum(), 
    how="left", 
    on="year").join(
    orders_agg.groupby(["year"])["products"].sum(), 
    how="left", 
    on="year").join(
    orders_agg.groupby(["year"])["quantity"].sum(), 
    how="left", 
    on="year")

,year,order_id,revenue,products,quantity
0,2014,969,483966.1261,1992,7579
1,2015,1038,470532.5090,2102,7979
2,2016,1315,609205.5980,2587,9837
3,2017,1687,733215.2552,3312,12476


In [22]:
# Total orders, sales, products, quantities, delivery timeline through 2014-2017 - QUARTERLY
orders_agg_yr_qtr = (orders_agg.groupby(["year","quarter"])["order_id"].count().reset_index()).join(
    orders_agg.groupby(["year","quarter"])["revenue"].sum(), 
    how="left", 
    on=["year","quarter"]).join(
    orders_agg.groupby(["year","quarter"])["products"].sum(), 
    how="left", 
    on=["year","quarter"]).join(
    orders_agg.groupby(["year","quarter"])["quantity"].sum(), 
    how="left", 
    on=["year","quarter"]).join(
    round( orders_agg.groupby(["year","quarter"])["days_till_shipped"].median(), 2), 
    how="left", 
    on=["year","quarter"])

orders_agg_yr_qtr.columns = ["year","quarter","orders","revenue","products","quantity","days_till_shipped"]
orders_agg_yr_qtr = orders_agg_yr_qtr[["year","quarter","orders","days_till_shipped","products","quantity","revenue"]]
x1 = orders_agg_yr_qtr.revenue.values[0]
change_rev = []
for x in orders_agg_yr_qtr.revenue:
    if(x==x1):
        change_rev.append(0)
    else:
        change_rev.append(round((x-x1)/x,2))
        x1 = x
orders_agg_yr_qtr["%_change_rev"] = change_rev
orders_agg_yr_qtr

,year,quarter,orders,days_till_shipped,products,quantity,revenue,%_change_rev
0,2014,1,185,59.0,385,1438,96498.7200,0.00
1,2014,2,208,6.0,405,1516,83355.5086,-0.16
2,2014,3,257,4.0,545,2097,139306.0173,0.40
3,2014,4,319,3.0,657,2528,164805.8802,0.15
4,2015,1,186,59.0,342,1301,90952.3496,-0.81
5,2015,2,248,5.0,488,1788,97852.8812,0.07
6,2015,3,275,4.0,588,2273,145554.2330,0.33
7,2015,4,329,4.0,684,2617,136173.0452,-0.07
8,2016,1,235,60.0,473,1782,136898.6390,0.01
9,2016,2,308,5.0,637,2394,149148.5428,0.08


- 2015 Q1 rev decreased by 81%, Q4 by 7%
- 2016 Q3 rev decreased by 14%
- 2017 Q2 rev decreased by 48%

In [45]:
# Total orders shipped for each ship mode through 2017
orders_agg[orders_agg["year"] == 2017].groupby(["ship_mode","month_num"])["order_id"].count().reset_index().pivot(
    index = "month_num",
    columns=["ship_mode"], 
    values = "order_id")

ship_mode,First Class,Same Day,Second Class,Standard Class
month_num,,,,
1,20,4,20,58
2,16,5,19,67
3,22,7,42,85
4,33,4,18,79
5,22,8,18,64
6,15,5,23,84
7,15,5,20,82
8,20,5,27,70
9,39,7,33,103


- Standard class ship mode - shipped highest number of orders in all three years, however same day orders shipped the lowest orders all through years

- Throughout the years, each ship modes showed a consistent delivery times approximately

In [47]:
# Avg days till orders shipped for each month through 2014-2017
round(orders_agg.groupby(["month_num","month","year"])["days_till_shipped"].median()).reset_index().pivot(
    index = ["month_num","month"],
    columns=["year"], 
    values = "days_till_shipped").reset_index()

year,month_num,month,2014,2015,2016,2017
0,1,January,76.0,90.0,60.0,45.0
1,2,February,120.0,59.0,60.0,89.0
2,3,March,5.0,6.0,6.0,19.0
3,4,April,5.0,5.0,6.0,5.0
4,5,May,6.0,7.0,6.0,4.0
5,6,June,6.0,5.0,4.0,4.0
6,7,July,6.0,5.0,5.0,5.0
7,8,August,5.0,4.0,4.0,4.0
8,9,September,4.0,4.0,4.0,4.0
9,10,October,2.0,2.0,4.0,2.0


- January and february consistently observed highest time till orders were shipped

In [ ]:
# Shipment time based on category
round(orders_agg.merge(data[["Order ID","Category"]].drop_duplicates(),
                 "left", 
                 left_on="order_id", 
                 right_on="Order ID").groupby(["Category","year"])["days_till_shipped"].mean(), 2).reset_index().pivot(
    index = "Category",
    columns=["year"], 
    values = "days_till_shipped")

year,2014,2015,2016,2017
Category,,,,
Furniture,4.52,14.36,4.00,8.42
Office Supplies,11.61,9.92,7.04,8.67
Technology,8.43,6.34,4.42,11.47


In [39]:
# Shipment based on category for each month through 2014-2017
round(orders_agg.merge(data[["Order ID","Category"]].drop_duplicates(),
                 "left", left_on="order_id", right_on="Order ID").groupby(["Category","month_num","month","year"])["days_till_shipped"].mean()).reset_index().pivot(
    index = ["Category","month_num","month"],
    columns=["year"], 
    values = "days_till_shipped").reset_index()

year,Category,month_num,month,2014,2015,2016,2017
0,Furniture,1,January,58.0,86.0,64.0,73.0
1,Furniture,2,February,97.0,89.0,77.0,78.0
2,Furniture,3,March,33.0,66.0,56.0,71.0
3,Furniture,4,April,38.0,46.0,43.0,30.0
4,Furniture,5,May,44.0,72.0,31.0,7.0
5,Furniture,6,June,48.0,29.0,18.0,22.0
6,Furniture,7,July,37.0,32.0,31.0,12.0
7,Furniture,8,August,29.0,10.0,-9.0,13.0
8,Furniture,9,September,-44.0,-1.0,-37.0,-9.0
9,Furniture,10,October,-40.0,-23.0,-38.0,-44.0
